# 12 — 全10群削除とLoRAの順序 × DE有無

Llama 3.2 3B / Gemma 3 4B、全7モジュール × 全10群 × DE2条件を、**学習→削除**と**削除→学習**の両方で比較する。第1群が最小、第10群が最大。各条件は同一モデルから独立に開始する。

今回は **WikiText-2 trainによるLoRA、新基準（BOS 1個＋本文2047個、非重複、末尾切捨て）でPPL評価**。BoolQの学習・accuracyは含めず、後日別実験とする。同一ドメインの汎化・回復性の評価であり、一般能力やalignmentの保証ではない。

- 学習→削除: 無削除モデルを1回学習し、LoRAをmergeしてから各行列のSVD/DEを再計算する。
- 削除→学習: 逆DE変換した削除済み重みにLoRA。各条件の学習は独立。削除した方向の再獲得を禁止しない（rank固定圧縮ではない）。
- 無削除・学習前/後のPPLを共通対照とする。学習後対照との差を集計する。
- 1モデル・1seed: **141回の学習、280条件の最終比較**。加えて削除→学習の140条件の学習前PPLと2種類のbaselineを評価。両モデルなら学習282回。
- 元の06/11は変更しない。本ノートブックは2026-11-06作成、本番未実行。

**2026-11-13変更:** 評価は2048・各ブロックBOS。学習は1024・BOSなしのまま。評価と学習の識別を分離。旧v1保存先は変更せず、新v2で管理（旧adapterの自動移行は行わない）。

現在の実行範囲は下の設定セルを参照。Llamaの削除→学習のみUP・GATEに限定しています。


## セットアップと実行方法
VS CodeでこのファイルをColab GPUカーネルに接続する。L4でも開始可能だが、Gemmaの学習・SVDのメモリと速度は実測して判断する。A100は高速化が必要な場合に選択。GPUでの本番動作は作成時には未検証。

初回は必要に応じて `INSTALL_DEPENDENCIES=True`。既存Colabのtorchを不用意に交換しない。HFキーは環境変数/非表示入力のみ。W&Bは今回は使わず、Driveに結果・adapter・再開用checkpointを保存する。


In [1]:
from pathlib import Path
import os, sys, subprocess, getpass
INSTALL_DEPENDENCIES = False  # 初回に必要な場合のみTrue。torch自体はColabのものを使う。
REPO = next((p for p in [Path.cwd(), *Path.cwd().resolve().parents, Path('/content/RMT_utils'), Path.home()/'GitHub/RMT_utils'] if (p/'funcs1.py').exists()), Path('/content/RMT_utils'))
if not (REPO/'funcs1.py').exists():
    subprocess.run(['git','clone','https://github.com/ryy1210/RMT_utils',str(REPO)],check=True)
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements-notebooks.txt'),'transformers==4.57.6'],check=True)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
if not os.environ.get('HF_TOKEN'):
    token=getpass.getpass('HF_TOKEN (ログイン済みなら空欄): ').strip()
    if token: os.environ['HF_TOKEN']=token
print('Repository:',REPO)

# 2026-11-13: paper protocol fixes the Transformers version.
import importlib.metadata
if importlib.metadata.version('transformers')!='4.57.6' or getattr(sys.modules.get('transformers'),'__version__','4.57.6')!='4.57.6':
    raise RuntimeError('INSTALL_DEPENDENCIES=Trueで設定セルを実行し、カーネルを再起動してください: transformers==4.57.6')

Repository: /content/RMT_utils


## 設定
`MODEL_KEY`で1モデルを選択。`SEEDS`、`PHASES`は実行範囲。両phase・全10群が既定。学習条件は既存06を踏襲。epoch数は固定し、test PPLによる早期停止や条件選択をしない。変更時は新しいOUTPUTを使う。

2026-11-15: Llamaの削除→学習はUP・GATEのみ（40条件）、学習→削除は全7モジュール（140条件）。
学習対象LoRAは両順序とも全7種類のまま。既存結果・adapterは同じ保存先から再利用する。
確認時点ではUPのDEなし第1〜8群が完了し、追加学習はUP残り12＋GATE20＝32条件。
Gemmaの削除→学習対象は13の結果後に設定する。自動解放はFalse（共有サーバー上の13を停止させない）。


In [2]:
import json, hashlib, math, gc, time, re, platform, importlib.metadata, inspect, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import funcs1
from funcs1 import _dyson_from_svd
if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT=Path('/content/drive/MyDrive/TUS/hashiguchi/data')
else:
    found=list((Path.home()/'Library/CloudStorage').glob('GoogleDrive-*/*/TUS/hashiguchi/data'))
    if len(found)!=1: raise RuntimeError('DATA_ROOTを手動設定してください: '+str(found))
    DATA_ROOT=found[0]
MODEL_KEY='llama'  # 'llama' または 'gemma'。必ず1モデルずつ実行する。
OUTPUT_ROOT=DATA_ROOT/'ibis2026_de_lra'/'decile-finetune-order-wikitext2-paper-v2'
OUTPUT=OUTPUT_ROOT/MODEL_KEY
LEGACY_OUTPUT=DATA_ROOT/'ibis2026_de_lra'/'ibis2026-de-controlled-v2-bulk-order'
MODELS={'llama':'meta-llama/Llama-3.2-3B','gemma':'google/gemma-3-4b-pt'}
RUN_MODELS=[MODEL_KEY]  # 内部用。通常はMODEL_KEYだけ変更する。
SEEDS=[42]  # 論文用の反復は例 [42,43,44]。seed別に保存される。
PHASES=['finetune_then_prune','prune_then_finetune']
GROUPS={'UP':['up_proj'],'DOWN':['down_proj'],'GATE':['gate_proj'],'Q':['q_proj'],'K':['k_proj'],'V':['v_proj'],'O':['o_proj']}
# 2026-11-15変更: 実行範囲だけを限定。GROUPS・LoRA対象・保存先・学習識別は維持する。
# Llamaの削除→学習はUP/GATE。Gemmaの選定は13の完了後に行う（未設定では開始しない）。
PRUNE_THEN_FINETUNE_GROUPS={'llama':['UP','GATE'], 'gemma':None}

def phase_groups(key, phase):
    if phase == 'finetune_then_prune':
        return list(GROUPS)
    selected=PRUNE_THEN_FINETUNE_GROUPS.get(key)
    if not selected:
        raise ValueError(f'{key}: 削除→学習の対象をPRUNE_THEN_FINETUNE_GROUPSで指定してください。')
    if len(set(selected)) != len(selected) or not set(selected) <= set(GROUPS):
        raise ValueError('削除→学習のモジュール指定が不正です。')
    return list(selected)

DECILES=list(range(1,11))  # 全10群。1が最小、10が最大。
REVISIONS={'llama':'13afe5124825b4f3751f836b40dafda64c1ed062','gemma':'cc012e0a6d0787b4adcc0fa2c4da74402494554d'}
TOTAL={'llama':3212749824,'gemma':3880263168}
TRAIN_SEQ_LEN=1024; EVAL_SEQ_LEN=2048; BATCH_SIZE=1; SVD_DEVICE='cuda'; TRAIN_BF16=True
TRAIN=dict(epochs=1,learning_rate=2e-4,batch=1,accumulation=8,rank=8,alpha=16,dropout=0.05,
           warmup_ratio=0.0,save_steps=100,logging_steps=10)
# 既存06と同じ基本LoRA条件。全7モジュールを学習。テストPPLで調整しない。
LOCAL_CACHE=Path('/content/rmt12_cache') if Path('/content').exists() else Path('/tmp/rmt12_cache')
CACHE_ROOT=LOCAL_CACHE
DELETE_CACHE_AFTER_GROUP=True
RUN_EXPERIMENTS=True  # 設定・軽量検証を確認後Trueにして本番セルを実行
AUTO_DISCONNECT=True  # 13とサーバー共有時に、他の実験を停止させないため自動解放しない。
ALL_REQUESTED_COMPLETED=False

# 評価だけ変更。学習は従来どおり1024・BOSなし・固定epoch。
TRAIN_TARGET_MODULES=['up_proj','down_proj','gate_proj','q_proj','k_proj','v_proj','o_proj']
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 共通のSVD・DE・PPL処理
既存11と同じ転置、昇順10分割、DE逆変換、Gemmaテキスト重みの検証付きロード、新基準2048・各ブロックBOSのPPL計算を使用する。

In [3]:
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def digest(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False,default=paper_json_default));tmp.replace(path)

def safe_exp(x): return math.exp(x) if x<709 else float('inf')

def atomic_csv(path,df):
    tmp=path.with_suffix('.tmp.csv');df.to_csv(tmp,index=False);tmp.replace(path)

def decompose(weight):
    transposed=weight.shape[0]>weight.shape[1]
    w=(weight.T if transposed else weight).float().to(SVD_DEVICE)
    u,s,v=torch.linalg.svd(w,full_matrices=False)
    # 既存DEと同じ推定式。GPU thin SVD結果をfloat64で渡す。
    y,x,z=_dyson_from_svd(w.cpu().double().numpy(),u.cpu().double().numpy(),s.cpu().double().numpy(),v.cpu().double().numpy())
    ud,sd,vd=torch.linalg.svd(torch.as_tensor(y,device=SVD_DEVICE,dtype=torch.float32),full_matrices=False)
    return {False:(u,s,v,None,None,transposed),True:(ud,sd,vd,x,z,transposed)}

def reconstruct(parts,indices):
    u,s,v,x,z,t=parts
    idx=torch.as_tensor(indices,device=s.device,dtype=torch.long)
    result=(u[:,idx]*s[idx])@v[idx,:]
    if x is not None:
        result*=torch.as_tensor(np.sqrt(x),device=result.device,dtype=result.dtype)[:,None]
        result*=torch.as_tensor(np.sqrt(z),device=result.device,dtype=result.dtype)[None,:]
    if t: result=result.T
    if not torch.isfinite(result).all(): raise FloatingPointError('Nonfinite reconstruction')
    return result

def check_loading_info(info):
    if not isinstance(info,dict): raise RuntimeError('Loading report missing')
    errors={k:info.get(k) for k in ['missing_keys','unexpected_keys','mismatched_keys','error_msgs'] if info.get(k)}
    if errors: raise RuntimeError(f'Pretrained checkpoint did not load completely: {errors}')

def extract_gemma_text(full):
    from transformers import Gemma3ForCausalLM
    # 正式クラスでロードした実Tensorを共有。meta上の空モデルへ全モジュールを置換する。
    decoder=full.model.language_model
    with torch.device('meta'):
        text=Gemma3ForCausalLM(full.config.text_config)
    text.model=decoder
    text.lm_head=full.lm_head
    text.generation_config=full.generation_config
    if any(p.is_meta for p in text.parameters()) or any(b.is_meta for b in text.buffers()):
        raise RuntimeError('Unloaded meta tensors remain')
    return text

def model_load(key):
    kwargs=dict(revision=REVISIONS[key],token=os.environ.get('HF_TOKEN'),torch_dtype=torch.bfloat16,
                low_cpu_mem_usage=True,output_loading_info=True,attn_implementation='sdpa')
    if key=='gemma':
        from transformers import Gemma3ForConditionalGeneration
        # 修正: checkpointのarchitectureと一致するクラスを使う。text_configだけを渡さない。
        full,info=Gemma3ForConditionalGeneration.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
        model=extract_gemma_text(full)
        del full; gc.collect()
    else:
        model,info=AutoModelForCausalLM.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
    actual=sum(p.numel() for p in model.parameters())
    if actual!=TOTAL[key]: raise ValueError(f'Parameter count mismatch {actual} != {TOTAL[key]}')
    if any(p.is_meta for p in model.parameters()): raise RuntimeError('Unloaded model parameters')
    root=result_root(key); root.mkdir(parents=True,exist_ok=True)
    write_json(root/'loading_report.json',dict(model=MODELS[key],revision=REVISIONS[key],
        loader_version=GEMMA_LOADER_VERSION if key=='gemma' else 'auto_verified_v2',
        transformers=importlib.metadata.version('transformers'),parameter_count=actual,
        missing_keys=[],unexpected_keys=[],mismatched_keys=[],config_hash=CONFIG_HASH))
    model.eval(); model.config.use_cache=False
    return model.to('cuda')

def evaluate(model,tokens):
    model.eval()
    losses=[]; counts=[]
    for start in range(0,len(tokens),BATCH_SIZE):
        ids=torch.as_tensor(tokens[start:start+BATCH_SIZE],device=next(model.parameters()).device)
        logits=model(input_ids=ids,use_cache=False).logits
        sums=torch.zeros(ids.shape[0],device=ids.device,dtype=torch.float64)
        for pos in range(0,ids.shape[1]-1,64):
            end=min(pos+64,ids.shape[1]-1)
            loss=F.cross_entropy(logits[:,pos:end].float().transpose(1,2),ids[:,pos+1:end+1],reduction='none')
            if not torch.isfinite(loss).all(): raise FloatingPointError('Nonfinite NLL')
            sums+=loss.double().sum(1)
        losses.extend(sums.cpu().tolist()); counts.extend([ids.shape[1]-1]*ids.shape[0]); del logits
        if start==0 or (start+1)%25==0 or start+1==len(tokens): print(f'  PPL blocks {start+1}/{len(tokens)}',flush=True)
    return np.asarray(losses),np.asarray(counts)

def resolve(model,name):
    found=[m for k,m in model.named_modules() if isinstance(m,torch.nn.Linear) and (k==name or k.endswith('.'+name))]
    if len(found)!=1: raise ValueError(f'Ambiguous module {name}')
    return found[0]

def function_fingerprint(f):
    # ipykernelの一時filename・セル番号は再起動で変わるのでhashに含めない。
    import types
    def normalize(code):
        return [code.co_code.hex(),code.co_names,code.co_varnames,
                [normalize(x) if isinstance(x,types.CodeType) else repr(x) for x in code.co_consts]]
    return digest(normalize(inspect.unwrap(f).__code__))

def read_result(path):
    path=evaluation_dir(path)  # 旧評価・新評価を混ぜない。学習の保存先は変えない。
    if not (path/'summary.json').exists(): return None
    obj=json.loads((path/'summary.json').read_text())
    assert obj['config_hash']==CONFIG_HASH
    f=path/'losses.npz'
    assert sha(f)==obj['loss_hash'], 'Result checksum mismatch'
    with np.load(f) as z: loss=z['loss'];count=z['count']
    assert len(loss)==len(TOKENS[obj['model']]) and np.all(count==EVAL_SEQ_LEN-1) and np.isfinite(loss).all()
    assert obj['eval_hash']==EVAL_HASH
    v=safe_exp(float(loss.sum()/count.sum()))
    assert (None if not np.isfinite(v) else v)==obj['ppl'], 'PPL mismatch'
    return obj

def save_result(path,key,loss,count,extra):
    path=evaluation_dir(path)
    path.mkdir(parents=True,exist_ok=True)
    with open(path/'losses.tmp','wb') as f: np.savez_compressed(f,loss=loss,count=count)
    (path/'losses.tmp').replace(path/'losses.npz')
    value=safe_exp(float(loss.sum()/count.sum()))
    obj=dict(model=key,config_hash=CONFIG_HASH,eval_hash=EVAL_HASH,protocol=PAPER_PROTOCOL,ppl=value if np.isfinite(value) else None,
             ppl_overflow=not np.isfinite(value),loss_hash=sha(path/'losses.npz'),**extra)
    write_json(path/'summary.json',obj)
    return obj

def decile_indices(p,decile):
    if p<10 or decile not in DECILES:raise ValueError('Need p>=10 and decile 1..10')
    # torch.linalg.svdは降順。最小側から順に群を割り当てる。
    return np.array_split(np.arange(p-1,-1,-1),10)[decile-1]

def remove_decile(original,parts,decile):
    idx=decile_indices(len(parts[1]),decile)
    removed=reconstruct(parts,idx)
    value=original.to(device=removed.device,dtype=torch.float32)-removed
    if not torch.isfinite(value).all():raise FloatingPointError('Nonfinite pruned weight')
    return value,idx

def cache_path(key,group,name):
    return CACHE_ROOT/CONFIG_HASH/key/group/(digest(name)[:24]+'.pt')

def prepare_svd_cache(key,group,matrices):
    import shutil
    root=CACHE_ROOT/CONFIG_HASH/key/group;root.mkdir(parents=True,exist_ok=True)
    required=sum(2*4*(min(m.weight.shape)*(sum(m.weight.shape)+1)+sum(m.weight.shape)) for _,m in matrices)
    existing=sum(p.stat().st_size for p in root.glob('*.pt'))
    if shutil.disk_usage(root).free < max(0,required-existing)+1024**3:
        raise RuntimeError(f'SVD cache requires about {required/1e9:.1f} GB; insufficient free disk')
    for i,(name,module) in enumerate(matrices):
        path=cache_path(key,group,name);receipt=path.with_suffix('.json')
        if path.exists() and receipt.exists():
            obj=json.loads(receipt.read_text())
            assert obj['config_hash']==CONFIG_HASH and obj['name']==name and obj['sha256']==sha(path)
            assert obj['shape']==list(module.weight.shape)
            continue
        print(f'{key} {group} SVD {i+1}/{len(matrices)}: {name}',flush=True)
        original=module.weight.detach().cpu().clone();parts=decompose(original)
        stored={}
        for de,(u,s,v,x,z,t) in parts.items():
            stored[de]=(u.cpu(),s.cpu(),v.cpu(),None if x is None else torch.as_tensor(np.asarray(x).copy()),
                        None if z is None else torch.as_tensor(np.asarray(z).copy()),t)
        tmp=path.with_suffix('.tmp');torch.save(stored,tmp);tmp.replace(path)
        write_json(receipt,dict(config_hash=CONFIG_HASH,name=name,shape=list(original.shape),sha256=sha(path)))
        del original,parts,stored;gc.collect();torch.cuda.empty_cache()
    return root

def load_parts(path,de,device):
    obj=torch.load(path,map_location='cpu',weights_only=True)
    u,s,v,x,z,t=obj[de]
    return u.to(device),s.to(device),v.to(device),None if x is None else x.numpy(),None if z is None else z.numpy(),t

def validate_group(model,targets):
    matrices=[(n,m) for n,m in model.named_modules() if isinstance(m,torch.nn.Linear) and n.split('.')[-1] in targets]
    for target in targets:
        names=[n for n,m in matrices if n.split('.')[-1]==target]
        blocks=[int(re.search(r'layers\.(\d+)\.',n).group(1)) for n in names]
        assert sorted(blocks)==list(range(model.config.num_hidden_layers)), f'Incomplete module: {target}'
    return matrices

def smoke_test():
    global SVD_DEVICE
    old=SVD_DEVICE;SVD_DEVICE='cpu'
    try:
        torch.manual_seed(17)
        for shape in [(24,80),(80,24),(24,24)]:
            w=torch.randn(*shape);parts=decompose(w);p=min(shape)
            bins=[decile_indices(p,d) for d in DECILES]
            assert np.array_equal(np.sort(np.concatenate(bins)),np.arange(p))
            assert max(map(len,bins))-min(map(len,bins))<=1
            assert bins[0][0]==p-1 and bins[-1][-1]==0
            for de in [False,True]:
                assert torch.allclose(reconstruct(parts[de],np.arange(p)),w,atol=4e-5,rtol=4e-5)
                for d,idx in zip(DECILES,bins):
                    got,removed=remove_decile(w,parts[de],d)
                    expected=reconstruct(parts[de],np.setdiff1d(np.arange(p),idx))
                    assert torch.allclose(got,expected,atol=4e-5,rtol=4e-5)
        print('PASS: 10群分割・逆変換・転置・全20条件の削除式')
    finally:SVD_DEVICE=old

evaluate=torch.inference_mode()(evaluate)
GEMMA_LOADER_VERSION="conditional_verified_v2"
def result_root(key): return OUTPUT


# Shared paper protocol, embedded in notebooks 12 and 13 for uncommitted Colab use.
PAPER_PROTOCOL = dict(version='paper-2048-each-bos-v1', length=2048, bos='each_block',
    packing='one BOS + 2047 text; disjoint; drop incomplete tail', batch=1,
    corpus_revision='b08601e04326c79dfdd32d625aee71d232d685c3', precision='bf16', attention='sdpa')

def pack_paper_tokens(raw, bos_id, length=2048):
    raw=np.asarray(raw,dtype=np.int64)
    if length<2 or bos_id is None: raise ValueError('Invalid BOS/length')
    n=len(raw)//(length-1)
    if not n: raise ValueError('No complete evaluation blocks')
    a=np.empty((n,length),dtype=np.int64);a[:,0]=bos_id
    a[:,1:]=raw[:n*(length-1)].reshape(n,length-1)
    return a,dict(blocks=n,length=length,scored_tokens=n*(length-1),dropped_tokens=len(raw)-n*(length-1),bos_token_id=int(bos_id))

def paper_environment():
    return dict(python=platform.python_version(),versions={k:importlib.metadata.version(k) for k in
        ('torch','transformers','datasets','numpy')},gpu=torch.cuda.get_device_name() if torch.cuda.is_available() else 'cpu',
        cuda=torch.version.cuda,attention='sdpa',tf32=False)

def paper_json_default(x):
    if isinstance(x,(set,frozenset)): return sorted(x,key=repr)
    if isinstance(x,np.generic):return x.item()
    if isinstance(x,np.ndarray):return x.tolist()
    if isinstance(x,(Path,torch.dtype,torch.device)):return str(x)
    raise TypeError(type(x).__name__)

def prepare_paper_inputs(key, data_root, model_id, revision, train_length=None):
    from transformers import AutoTokenizer
    from datasets import load_dataset
    root=data_root/'ibis2026_de_lra/paper-2048-each-bos-inputs-v1';root.mkdir(parents=True,exist_ok=True)
    tok=None; results={}
    for split in (['test','train'] if train_length is not None else ['test']):
        length=2048 if split=='test' else train_length
        identity=dict(model=model_id,revision=revision,split=split,length=length,
            dataset_revision=PAPER_PROTOCOL['corpus_revision'],transformers=importlib.metadata.version('transformers'),
            packing=PAPER_PROTOCOL['packing'] if split=='test' else 'no BOS; disjoint full blocks; drop tail')
        p=root/f'{key}_{split}_{digest(identity)[:16]}.npy';receipt=p.with_suffix('.json')
        if p.exists() and receipt.exists():
            meta=json.loads(receipt.read_text())
            if meta['identity']!=identity or meta['sha256']!=sha(p):raise ValueError('Input cache checksum/identity mismatch')
            a=np.load(p,allow_pickle=False)
        else:
            text_path=root/f'{split}.txt'
            if not text_path.exists():
                old=data_root/'ibis2026_de_lra/gemma-qk-ppl-settings-grid-v1/inputs/wikitext2_test.txt'
                if split=='test' and old.exists():text=old.read_text(encoding='utf-8')
                else:
                    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split,revision=PAPER_PROTOCOL['corpus_revision'])
                    text='\n\n'.join(ds['text'])
                text_path.write_text(text,encoding='utf-8')
            if split=='test' and sha(text_path)!='696cca6b65a171b0a358a4be6732cdfdf2dd6164a32e20fd70e3c13fc4dfae83':
                raise ValueError('Test corpus differs from audited 11_1')
            if tok is None:tok=AutoTokenizer.from_pretrained(model_id,revision=revision,token=os.environ.get('HF_TOKEN'))
            raw=np.asarray(tok(text_path.read_text(encoding='utf-8'),add_special_tokens=False)['input_ids'],dtype=np.int64)
            if split=='test':a,meta=pack_paper_tokens(raw,tok.bos_token_id,length)
            else:
                a=raw[:len(raw)//length*length].reshape(-1,length).copy()
                if not len(a):raise ValueError('Empty training data')
                meta=dict(blocks=len(a),length=length,dropped_tokens=len(raw)%length)
            with p.with_suffix('.tmp').open('wb') as f:np.save(f,a,allow_pickle=False)
            p.with_suffix('.tmp').replace(p)
            meta.update(identity=identity,sha256=sha(p),text_sha256=sha(text_path));write_json(receipt,meta)
        if a.ndim!=2 or a.shape[1]!=length or not len(a):raise ValueError('Token shape mismatch')
        if split=='test' and not np.all(a[:,0]==meta['bos_token_id']):raise ValueError('BOS missing')
        results[split]=(a,meta)
    return results

def evaluation_identity(token_hashes):
    # Evaluation identity is separate from training: new evaluation never invalidates adapters.
    return digest(dict(protocol=PAPER_PROTOCOL,tokens=token_hashes,environment=paper_environment(),
        evaluate=function_fingerprint(evaluate),pack=function_fingerprint(pack_paper_tokens)))

def evaluation_dir(path):return Path(path)/'evaluations'/EVAL_HASH

## 学習・チェックポイント・条件別再開
完了結果はチェックサムで検証し、全完了モデルはロードしない。中断した学習はoptimizer/RNGを含む検証済みcheckpointから再開する。チェックポイントがなければその条件の学習を最初からやり直す。学習後adapterだけ完了している場合は再学習せず評価を再開する。

SVDは学習前後・seed・adapterごとに別キャッシュ。完了したモジュールの一時SVDは削除する。Colabが消えると一時キャッシュは消えるが、Drive上の完了結果・学習は保持される。部分完了モジュールで必要なSVDだけ再計算する。Driveの空き容量に注意（多数のadapter/optimizerを保持）。同じOUTPUTを複数カーネルから同時実行しない。


In [4]:
# 12専用: 既存11の削除・評価方式を維持し、学習順序と再開管理を追加。
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import Trainer, TrainingArguments, TrainerCallback, set_seed
from torch.utils.data import Dataset
import shutil

class PackedTokens(Dataset):
    def __init__(self, array): self.array = array
    def __len__(self): return len(self.array)
    def __getitem__(self, i):
        ids = torch.tensor(self.array[i], dtype=torch.long)
        return {'input_ids': ids, 'attention_mask': torch.ones_like(ids), 'labels': ids.clone()}

def snapshot_files(root):
    return {str(p.relative_to(root)): sha(p) for p in sorted(root.rglob('*'))
            if p.is_file() and p.name != 'receipt.json'}

def verify_files(root, receipt):
    for rel, checksum in receipt['files'].items():
        p=root/rel
        if not p.is_file() or sha(p)!=checksum:
            raise RuntimeError(f'Incomplete/corrupt checkpoint: {p}')

def training_receipt(path):
    f=path/'adapter'/'receipt.json'
    if not f.exists(): return None
    obj=json.loads(f.read_text())
    if obj['config_hash']!=CONFIG_HASH: raise ValueError('Training configuration mismatch')
    verify_files(path/'adapter',obj)
    return obj

class CompleteCheckpoint(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        root=Path(args.output_dir)/f'checkpoint-{state.global_step}'
        # receiptを最後に書く。転送途中・カーネル切断時のcheckpointを完了扱いしない。
        write_json(root/'receipt.json',dict(config_hash=CONFIG_HASH,step=state.global_step,files=snapshot_files(root)))

def resume_checkpoint(path):
    checkpoints=sorted((path/'trainer').glob('checkpoint-*'),key=lambda p:int(p.name.split('-')[-1]),reverse=True)
    for root in checkpoints:
        receipt=root/'receipt.json'
        if not receipt.exists():
            print('Incomplete checkpoint ignored:',root.name);continue
        obj=json.loads(receipt.read_text())
        if obj['config_hash']!=CONFIG_HASH: raise ValueError('Checkpoint configuration mismatch')
        try: verify_files(root,obj)
        except RuntimeError:
            print('Corrupt checkpoint ignored:',root.name);continue
        return str(root)
    return None

def train_or_merge(model,key,seed,path):
    receipt=training_receipt(path)
    if receipt:
        print('SKIP training / load adapter:',path,flush=True)
        merged=PeftModel.from_pretrained(model,str(path/'adapter'),is_trainable=False).merge_and_unload()
        return merged.eval()
    set_seed(seed)
    model.config.use_cache=False
    model=get_peft_model(model,LoraConfig(task_type=TaskType.CAUSAL_LM,r=TRAIN['rank'],
        lora_alpha=TRAIN['alpha'],lora_dropout=TRAIN['dropout'],bias='none',
        target_modules=TRAIN_TARGET_MODULES))
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    if trainable<=0: raise RuntimeError('No trainable LoRA parameters')
    print('LoRA trainable parameters:',trainable,flush=True)
    model.enable_input_require_grads()
    args=TrainingArguments(output_dir=str(path/'trainer'),num_train_epochs=TRAIN['epochs'],
        learning_rate=TRAIN['learning_rate'],per_device_train_batch_size=TRAIN['batch'],
        gradient_accumulation_steps=TRAIN['accumulation'],warmup_ratio=TRAIN['warmup_ratio'],
        lr_scheduler_type='linear',weight_decay=0.0,optim='adamw_torch',bf16=TRAIN_BF16,
        fp16=False,use_cpu=not TRAIN_BF16,gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},
        save_strategy='steps',save_steps=TRAIN['save_steps'],save_total_limit=2,
        logging_steps=TRAIN['logging_steps'],report_to='none',dataloader_num_workers=0,
        dataloader_pin_memory=TRAIN_BF16,seed=seed,data_seed=seed,remove_unused_columns=False)
    trainer=Trainer(model=model,args=args,train_dataset=PackedTokens(TRAIN_TOKENS[key]),callbacks=[CompleteCheckpoint()])
    trainer.train(resume_from_checkpoint=resume_checkpoint(path))
    # Adapterのみを保存。学習→削除ではmerge後の実重みからSVD/DEを再計算する。
    temporary=path/'adapter.tmp'
    if temporary.exists(): shutil.rmtree(temporary)
    trainer.model.save_pretrained(temporary,safe_serialization=True)
    write_json(temporary/'receipt.json',dict(config_hash=CONFIG_HASH,seed=seed,
        trainable=trainable,global_step=trainer.state.global_step,files=snapshot_files(temporary)))
    final=path/'adapter'
    if final.exists(): raise RuntimeError(f'Unexpected unfinished adapter: {final}')
    temporary.rename(final)
    merged=trainer.model.merge_and_unload()
    merged.gradient_checkpointing_disable()
    merged.config.use_cache=False
    del trainer
    return merged.eval()

def experiment_base(key,seed): return OUTPUT/f'seed-{seed}'
def job_path(key,seed,phase,group,de,decile):
    return experiment_base(key,seed)/phase/group/f'de{int(de)}_decile{decile:02d}'

def complete_job(key,seed,phase,group,de,decile):
    path=job_path(key,seed,phase,group,de,decile)
    obj=read_result(path)
    if obj:
        expected=dict(model=key,seed=seed,phase=phase,group=group,de=de,decile=decile)
        if any(obj.get(k)!=v for k,v in expected.items()): raise ValueError('Result identity mismatch')
        if sha(path/'matrices.json')!=obj['matrix_plan_hash']: raise ValueError('Matrix plan checksum mismatch')
    return obj

def evaluate_save(model,key,path,extra):
    previous=read_result(path)
    if previous: return previous
    model.eval()
    loss,count=evaluate(model,TOKENS[key])
    return save_result(path,key,loss,count,extra)

def set_stage_cache(key,seed,stage,source_hash):
    global CACHE_ROOT
    # 同じ設定でも学習前後・seed・学習済みadapterが違えばSVDは共有しない。
    CACHE_ROOT=LOCAL_CACHE/key/f'seed-{seed}'/stage/source_hash

def prune_group(model,key,group,de,decile,path):
    matrices=validate_group(model,GROUPS[group])
    prepare_svd_cache(key,group,matrices)
    originals={name:m.weight.detach().cpu().clone() for name,m in matrices}
    records=[]
    try:
        for name,module in matrices:
            parts=load_parts(cache_path(key,group,name),de,module.weight.device)
            approx,idx=remove_decile(originals[name],parts,decile)
            with torch.no_grad(): module.weight.copy_(approx.to(module.weight.dtype))
            records.append(dict(name=name,shape=list(module.weight.shape),removed=len(idx),p=len(parts[1]),
                indices=idx.tolist(),pruned_weight_sha256=hashlib.sha256(module.weight.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes()).hexdigest(),relative_frobenius_error=float(torch.linalg.vector_norm(approx.cpu()-originals[name].float())/torch.linalg.vector_norm(originals[name].float()))))
            del parts,approx
        plan=path/'matrices.json'
        if plan.exists():
            prior=json.loads(plan.read_text())
            signature=lambda rows:[(r['name'],r['indices'],r['pruned_weight_sha256']) for r in rows]
            if prior['config_hash']!=CONFIG_HASH or signature(prior['records'])!=signature(records):
                raise RuntimeError('再開時の削除済み重みが以前と一致しません。既存adapterに異なるbaseを混ぜないため停止します。')
        else:
            write_json(plan,dict(config_hash=CONFIG_HASH,records=records))
        return originals,dict(matrix_count=len(records),sum_removed=sum(r['removed'] for r in records),
            matrix_plan_hash=sha(path/'matrices.json'))
    except BaseException:
        restore_group(model,originals)
        raise

def restore_group(model,originals):
    with torch.no_grad():
        for name,weight in originals.items(): resolve(model,name).weight.copy_(weight.to(next(model.parameters()).device))

def release():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def cleanup_group(key,group):
    p=CACHE_ROOT/CONFIG_HASH/key/group
    if DELETE_CACHE_AFTER_GROUP and p.exists(): shutil.rmtree(p)

def run12():
    global ALL_REQUESTED_COMPLETED
    ALL_REQUESTED_COMPLETED=False
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported(): raise RuntimeError('BF16対応CUDA GPUを選択してください')
    for key in RUN_MODELS:
      for seed in SEEDS:
        todo={phase:{group:[(de,d) for de in (False,True) for d in DECILES
            if complete_job(key,seed,phase,group,de,d) is None] for group in phase_groups(key,phase)} for phase in PHASES}
        base=experiment_base(key,seed)
        raw_baseline=read_result(base/'baseline_raw')
        ft_baseline=read_result(base/'baseline_ft')
        if raw_baseline and ft_baseline and not any(j for groups in todo.values() for j in groups.values()):
            print(key,seed,'all complete: skip model/SVD/training');continue
        # 全条件共通の無削除学習モデル。削除→学習のみを選んだ場合にも対照として必要。
        model=model_load(key)
        try:
            raw_baseline=evaluate_save(model,key,base/'baseline_raw',dict(kind='baseline_raw',seed=seed))
            if raw_baseline['ppl'] is None or raw_baseline['ppl']>1000: raise RuntimeError('Baseline異常: 学習を開始しません')
            need_ft=ft_baseline is None or any(todo.get('finetune_then_prune',{}).values())
            if need_ft:
                model=train_or_merge(model,key,seed,base/'shared_training')
                ft_baseline=evaluate_save(model,key,base/'baseline_ft',dict(kind='baseline_ft',seed=seed))
                source_hash=sha(base/'shared_training/adapter/receipt.json')
                set_stage_cache(key,seed,'finetuned',source_hash)
                for group,remaining in todo.get('finetune_then_prune',{}).items():
                    if not remaining: print('SKIP',key,group,'finetune_then_prune');continue
                    for de,d in remaining:
                        path=job_path(key,seed,'finetune_then_prune',group,de,d)
                        print('RUN',key,seed,'finetune_then_prune',group,de,d,flush=True)
                        originals,info=prune_group(model,key,group,de,d,path)
                        try:
                            evaluate_save(model,key,path,dict(phase='finetune_then_prune',group=group,de=de,decile=d,seed=seed,
                                baseline_ppl=ft_baseline['ppl'],raw_baseline_ppl=raw_baseline['ppl'],**info))
                        finally: restore_group(model,originals);del originals;release()
                    cleanup_group(key,group)
        finally: del model;release()
        set_stage_cache(key,seed,'pretrained',REVISIONS[key])
        for group,remaining in todo.get('prune_then_finetune',{}).items():
            if not remaining: print('SKIP',key,group,'prune_then_finetune');continue
            for de,d in remaining:
                path=job_path(key,seed,'prune_then_finetune',group,de,d)
                print('RUN',key,seed,'prune_then_finetune',group,de,d,flush=True)
                model=model_load(key)
                try:
                    # 完了adapterのロード時にも、元の削除済み重みを厳密に復元する。
                    originals,info=prune_group(model,key,group,de,d,path)
                    del originals;release()
                    initial=evaluate_save(model,key,path/'before_training',dict(kind='pruned_before_training',seed=seed,group=group,de=de,decile=d))
                    model=train_or_merge(model,key,seed,path/'training')
                    evaluate_save(model,key,path,dict(phase='prune_then_finetune',group=group,de=de,decile=d,seed=seed,
                        baseline_ppl=ft_baseline['ppl'],raw_baseline_ppl=raw_baseline['ppl'],ppl_before_training=initial['ppl'],**info))
                finally: del model;release()
            cleanup_group(key,group)
        collect_results()
    ALL_REQUESTED_COMPLETED=all(complete_job(k,s,p,g,de,d) is not None
        for k in RUN_MODELS for s in SEEDS for p in PHASES for g in phase_groups(k,p) for de in (False,True) for d in DECILES)
    if not ALL_REQUESTED_COMPLETED: raise RuntimeError('未完了条件があります')

def collect_results():
    rows=[]
    for k in RUN_MODELS:
      for s in SEEDS:
       for p in PHASES:
        for g in phase_groups(k,p):
         for de in (False,True):
          for d in DECILES:
           obj=complete_job(k,s,p,g,de,d)
           rows.append(dict(**obj,status='complete') if obj else dict(model=k,seed=s,phase=p,group=g,de=de,decile=d,status='pending',ppl=None))
    df=pd.DataFrame(rows)
    if 'baseline_ppl' in df: df['ppl_increase']=pd.to_numeric(df.ppl,errors='coerce')-df.baseline_ppl
    folder=evaluation_dir(OUTPUT);folder.mkdir(parents=True,exist_ok=True)
    atomic_csv(folder/'results.csv',df)
    return df

## 入力と設定の固定
trainはHugging Faceのdataset revisionとtokenizer revisionを固定して保存。testは11_1と同じ本文から新基準の2048・各ブロックBOSを生成。train/testは別split。validationによる探索は今回は行わない。ライブラリ/実装/入力/設定が変わると再開時に停止する。

In [5]:
def prepare_data():
    OUTPUT.mkdir(parents=True,exist_ok=True)
    arrays={};test_hash={};train_arrays={};train_receipts={}
    for key in RUN_MODELS:
        inputs=prepare_paper_inputs(key,DATA_ROOT,MODELS[key],REVISIONS[key],TRAIN_SEQ_LEN)
        arrays[key],meta=inputs['test'];test_hash[key]=meta['sha256']
        train_arrays[key],train_receipts[key]=inputs['train']
    return arrays,test_hash,train_arrays,train_receipts

def initialize12():
    global TOKENS,TOKEN_HASH,TRAIN_TOKENS,CONFIG_HASH,EVAL_HASH
    if MODEL_KEY not in MODELS or RUN_MODELS!=[MODEL_KEY]: raise ValueError('MODEL_KEYはllamaまたはgemmaを1つ指定してください')
    if not SEEDS or len(set(SEEDS))!=len(SEEDS) or not all(isinstance(s,int) and s>=0 for s in SEEDS): raise ValueError('SEEDS')
    if not PHASES or len(set(PHASES))!=len(PHASES) or not set(PHASES)<={'finetune_then_prune','prune_then_finetune'}: raise ValueError('PHASES')
    for key in RUN_MODELS:
        for phase in PHASES: phase_groups(key,phase)  # 入力取得前に実行範囲を検証
    TOKENS,TOKEN_HASH,TRAIN_TOKENS,train_receipts=prepare_data()
    # 学習・削除の識別にtest評価方式を含めない。run範囲を増やしても完了学習を再利用。
    core_names=['model_load','decompose','reconstruct','remove_decile','decile_indices','train_or_merge','prune_group']
    settings=dict(version='12-training-v2',model=MODELS[MODEL_KEY],revision=REVISIONS[MODEL_KEY],
        train=TRAIN,train_tokens=train_receipts,train_seq_len=TRAIN_SEQ_LEN,
        train_targets=TRAIN_TARGET_MODULES,dtype='bfloat16',training_bf16=TRAIN_BF16,svd_device=SVD_DEVICE,
        funcs1_sha256=sha(Path(funcs1.__file__)),implementation={x:function_fingerprint(globals()[x]) for x in core_names},
        packages={p:importlib.metadata.version(p) for p in ['torch','transformers','peft','datasets','numpy','accelerate']})
    CONFIG_HASH=digest(settings);EVAL_HASH=evaluation_identity(TOKEN_HASH)
    p=OUTPUT/'training_manifest.json'
    if p.exists() and json.loads(p.read_text())['config_hash']!=CONFIG_HASH:
        raise ValueError('学習設定/実装が変わりました。新しいOUTPUTを使用してください。')
    write_json(p,dict(config_hash=CONFIG_HASH,settings=settings))
    write_json(evaluation_dir(OUTPUT)/'manifest.json',dict(eval_hash=EVAL_HASH,protocol=PAPER_PROTOCOL,tokens=TOKEN_HASH,environment=paper_environment()))
    print('train blocks:',{k:len(v) for k,v in TRAIN_TOKENS.items()})
    print('test blocks:',{k:len(v) for k,v in TOKENS.items()})
    for phase in PHASES:
        groups=phase_groups(MODEL_KEY,phase)
        print(phase, '対象:', ', '.join(groups), '条件数/seed:', len(groups)*2*len(DECILES))
    display(collect_results().groupby(['model','seed','phase','status']).size().rename('conditions').reset_index())

initialize12()

train blocks: {'llama': 2379}
test blocks: {'llama': 141}
finetune_then_prune 対象: UP, DOWN, GATE, Q, K, V, O 条件数/seed: 140
prune_then_finetune 対象: UP, GATE 条件数/seed: 40


KeyboardInterrupt: 

## 軽量チェック

In [ ]:
# 実データ・事前学習済みモデルを使わない小行列チェック。
smoke_test()
print('本番は RUN_EXPERIMENTS=True の場合のみ次のセルで開始します。')

PASS: 10群分割・逆変換・転置・全20条件の削除式
本番は RUN_EXPERIMENTS=True の場合のみ次のセルで開始します。


## 本番
設定セルの `RUN_EXPERIMENTS=True` にして実行。再開も同じ設定で「すべて実行」。Gemmaは `MODEL_KEY="gemma"` に変更して実行する。保存先もモデル別に切り替わる。失敗を握り潰さず停止し、保存済み条件は次回スキップする。

In [ ]:
if RUN_EXPERIMENTS:
    run12()
else:
    print("本番未実行。RUN_EXPERIMENTS=Trueで開始。")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 MiB. GPU 0 has a total capacity of 22.03 GiB of which 7.69 MiB is free. Process 2779 has 8.90 GiB memory in use. Process 4055 has 8.60 GiB memory in use. Process 20837 has 3.02 GiB memory in use. Including non-PyTorch memory, this process has 1.48 GiB memory in use. Of the allocated memory 1.30 GiB is allocated by PyTorch, and 5.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 集計・比較図
横7モジュール、縦2順序、色でDEを比較。PPL増分は無削除・学習済みbaselineとの差。損失値はPPL計算/監査用にのみ保存。DEの差は負ほどDE有利。10群の曲線は独立条件であり累積削除ではない。

In [ ]:
RESULTS=collect_results()
display(RESULTS.groupby(['model','seed','phase','status']).size().rename('conditions').reset_index())
done=RESULTS[RESULTS.status=='complete'].copy()
if len(done):
    done['ppl']=pd.to_numeric(done.ppl,errors='coerce')
    pairs=done.pivot(index=['model','seed','phase','group','decile'],columns='de',values='ppl').reindex(columns=[False,True])
    pairs.columns=['ppl_no_de','ppl_de'];pairs=pairs.reset_index()
    pairs['de_minus_no_de']=pairs.ppl_de-pairs.ppl_no_de
    atomic_csv(evaluation_dir(OUTPUT)/'paired.csv',pairs);display(pairs)
    if done.ppl.isna().any(): print('PPL overflowあり。欠損を改善とみなさないでください。')
    for key in RUN_MODELS:
      for seed in SEEDS:
        fig,axes=plt.subplots(2,7,figsize=(25,7),squeeze=False)
        for row,phase in enumerate(['finetune_then_prune','prune_then_finetune']):
          for col,group in enumerate(GROUPS):
            ax=axes[row,col]
            if phase not in PHASES or group not in phase_groups(key,phase):
                ax.text(0.5,0.5,'Not selected',ha='center',va='center',transform=ax.transAxes)
                ax.set_title(group);ax.set_axis_off();continue
            for de in (False,True):
                g=done[(done.model==key)&(done.seed==seed)&(done.phase==phase)&(done.group==group)&(done.de==de)].set_index('decile').reindex(DECILES)
                ax.plot(DECILES,g.ppl,'-o' if not de else '--s',label='DE on' if de else 'DE off')
            for kind,color,style in [('baseline_raw','gray',':'),('baseline_ft','black','--')]:
                base=read_result(experiment_base(key,seed)/kind)
                if base and base['ppl'] is not None: ax.axhline(base['ppl'],color=color,ls=style,label=kind)
            ax.set(title=group,xlabel='Decile (1: smallest)',ylabel=phase+' / PPL',yscale='log',xticks=[1,5,10])
        axes[0,0].legend(fontsize=7);fig.suptitle(f'{key} seed={seed}');fig.tight_layout()
        fig.savefig(evaluation_dir(OUTPUT)/f'{key}_seed{seed}_order_ppl.pdf');plt.show()
else: print('まだ実行結果はありません。')

,model,seed,phase,status,conditions
0,llama,42,finetune_then_prune,pending,140
1,llama,42,prune_then_finetune,pending,140


まだ実行結果はありません。


## 正常終了後のみ切断

In [ ]:
if AUTO_DISCONNECT and RUN_EXPERIMENTS and ALL_REQUESTED_COMPLETED:
    from google.colab import runtime
    runtime.unassign()

In [ ]:
AUTO_DISCONNECT=True
if AUTO_DISCONNECT and RUN_EXPERIMENTS and ALL_REQUESTED_COMPLETED:
    from google.colab import runtime
    runtime.unassign()